# Regional Analysis — Freedom in the World

**Notebook 06 of 08**

### Purpose

Notebook 05 compared specific groups (EAC, major powers, Africa's leaders). This notebook steps up to **regions**: how do the world's five main regions differ in freedom, how have they changed since 2013, and what happens inside Africa when we zoom into its five sub-regions?

### Scope

The dataset supplies no regional classification, so the regions come from the **UN M49** geographic scheme (documented, user-approved for this notebook), mapped onto the dataset's own economy labels in `src/regions.py`. Nothing is classified by assumption.

## Data and Inputs

| Item | Location |
|---|---|
| Long analytical dataset | `data/processed/freedom_in_world_long.csv` |
| UN M49 classifications | `src/regions.py` |

**Question this notebook answers:** *How do the world's regions differ, how have their patterns changed, and what do Africa's sub-regions look like?*

### The classifications (documented, approved)

| Scheme | Groups | Source |
|---|---|---|
| UN M49 main regions | Africa (54), Americas (36), Asia (49), Europe (44), Oceania (14) — 197 economies | unstats.un.org/unsd/methodology/m49 |
| UN M49 Africa sub-regions | Eastern (18), Middle (9), Northern (6), Southern (5), Western (16) — 54 economies | same |

Notes recorded in `src/regions.py`: Cyprus is classified to Western Asia per UN M49; Kosovo (not a UN member) is assigned to Southern Europe per common M49-based usage; territories carried by the dataset (Puerto Rico, Hong Kong SAR) sit in their M49 region.

## Setup: imports and the project root

Same bootstrap. New imports: the UN M49 region lookups from `src/regions.py`.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd
import plotly.express as px

from src.data_loader import load_processed_data
from src.visualizations import create_multi_trend_chart, create_heatmap
from src.regions import get_region_lookup, get_africa_subregion_lookup, AFRICA_UN_M49

print('Imports ready.')

Imports ready.


### Interpretation

Setup ran cleanly. The region lookups map every dataset economy to exactly one UN M49 region — the next section proves it before any chart is drawn.

## 1. Load the overall-score series

**Question:** what data do the regional comparisons use?

**Method:** load the long dataset and keep `FH_FIW_TOTAL` with numeric scores (same preparation as notebook 05).

In [2]:
long = load_processed_data()
total = long[long['INDICATOR'] == 'FH_FIW_TOTAL'].copy()
total['Score'] = pd.to_numeric(total['Score'], errors='coerce')

print('TOTAL series:', total.shape)
print('Economies covered:', total['Economy'].nunique())

TOTAL series: (2758, 9)
Economies covered: 197


### Interpretation

2758 rows of overall scores. Each economy gets a region in the next step, then every aggregation below is a region-level mean.

## 2. Apply the region mapping

**Question:** does the UN M49 lookup cover the whole dataset?

**Method:** attach the main-region column and verify that all 197 economies are classified exactly once.

In [3]:
region_lookup = get_region_lookup()
total['region'] = total['Economy'].map(region_lookup)

print('Economies with a region:', total['region'].notna().sum(), 'of', total['Economy'].nunique())
print()
print('Economies per region:')
print(total[['region', 'Economy']].drop_duplicates()['region'].value_counts().to_string())

Economies with a region: 2758 of 197

Economies per region:
region
Africa      54
Asia        49
Europe      44
Americas    36
Oceania     14


### Interpretation

Every one of the 197 economies maps to exactly one region: **Africa 54, Asia 49, Europe 44, Americas 36, Oceania 14**. The classification is complete — nothing is left out and nothing is double-assigned.

## 3. Question 1 + 3 — current standing per region

**Question:** which regions have the highest and lowest scores in 2026?

**Method:** mean overall score per region for the latest year.

In [4]:
region_2026 = total[total['Year'] == 2026].groupby('region')['Score'].mean().round(2).sort_values()
print('Region means, 2026:')
print(region_2026.to_string())

fig = px.bar(
    region_2026,
    orientation='h',
    title='Mean overall score by region, 2026',
    labels={'value': 'Mean overall score (0-100)', 'region': 'Region'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Region means, 2026:
region
Asia        37.14
Africa      38.56
Americas    70.74
Europe      81.70
Oceania     84.21


### Interpretation

The world splits into two tiers: **Oceania (84.2), Europe (81.7) and the Americas (70.7)** sit far above **Asia (37.1) and Africa (38.6)** — a gap of roughly 35 points. Notice that Africa's mean (38.6) is pulled down by its many low scorers despite its leaders scoring 68–92 (notebook 05): the region is the most internally unequal.

## 4. Question 2 — regional trends

**Question:** how have the regions evolved since 2013?

**Method:** one line per region, mean overall score per year.

In [5]:
region_means = total.groupby(['region', 'Year'])['Score'].mean().reset_index()

fig = create_multi_trend_chart(
    region_means,
    title='Mean overall score by region, 2013-2026',
    y_label='Mean overall score (0-100)',
    group_col='region',
)
fig.show()

### Interpretation

**Oceania is the only region whose average rose** (81.6 → 84.2, +2.6) — the one bright line on the chart. Every other region declined: **Africa the most (−6.5, 45.0 → 38.6)**, then the Americas (−4.8), Asia (−4.4) and Europe (−3.1). The global decline from notebook 04 is a regional phenomenon with a single exception.

## 5. Question 4 — greatest changes and distributions

**Question:** which regions changed most, and how wide is the spread within each?

**Method:** plot the 2013→2026 change per region, then box plots of the 2026 country scores per region.

In [6]:
r2013 = total[total['Year'] == 2013].groupby('region')['Score'].mean()
r2026 = total[total['Year'] == 2026].groupby('region')['Score'].mean()
region_change = (r2026 - r2013).round(2).sort_values()
print('Change 2013 to 2026:')
print(region_change.to_string())

fig = px.bar(
    region_change,
    orientation='h',
    title='Change in mean overall score by region, 2013 to 2026',
    labels={'value': 'Change in score', 'region': 'Region'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Change 2013 to 2026:
region
Africa     -6.46
Americas   -4.84
Asia       -4.39
Europe     -3.14
Oceania     2.64


In [7]:
total_2026 = total[total['Year'] == 2026]
fig = px.box(
    total_2026,
    x='region',
    y='Score',
    title='Distribution of 2026 overall scores by region',
    labels={'Score': 'Overall score (0-100)', 'region': 'Region'},
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

The change chart confirms the trend: Africa fell furthest (−6.5), Europe least (−3.1), Oceania rose (+2.6). The box plots add the within-region story: **Europe and Oceania are compact and high**, the Americas wider but still high, while **Asia and Africa are both low and extremely wide** — Africa's 2026 scores range from 0 to 92, the largest spread of any region. Regional means genuinely summarize very different internal situations.

## 6. Zoom: Africa's sub-regions — trends

**Question:** which parts of Africa drove the region's decline?

**Method:** apply the UN M49 Africa sub-region lookup and plot the five sub-regions' means over time. Eastern Africa is where the EAC (notebook 05) sits.

In [8]:
africa_total = total[total['Economy'].isin(AFRICA_UN_M49)].copy()
africa_total['subregion'] = africa_total['Economy'].map(get_africa_subregion_lookup())

subregion_means = africa_total.groupby(['subregion', 'Year'])['Score'].mean().reset_index()
fig = create_multi_trend_chart(
    subregion_means,
    title='Mean overall score, Africa sub-regions, 2013-2026',
    y_label='Mean overall score (0-100)',
    group_col='subregion',
)
fig.show()

### Interpretation

All five African sub-regions declined, but very unevenly: **Southern Africa leads throughout** (64.8 → 62.6) and **Western Africa holds second** (52.7 → 48.4), while **Northern Africa collapsed from 38.0 to 23.2** — the sharpest fall on the continent — and Middle Africa fell to 23.6. **Eastern Africa**, home of the EAC, declined from 41.9 to 35.8, consistent with notebook 05's regional picture.

## 7. Africa's sub-regions — standing and movers

**Question:** what is the 2026 sub-regional ranking, and who changed most?

**Method:** rank by 2026 mean and plot the change since 2013.

In [9]:
sub_2026 = africa_total[africa_total['Year'] == 2026].groupby('subregion')['Score'].mean().round(2).sort_values()
print('Sub-region means, 2026:')
print(sub_2026.to_string())

fig = px.bar(
    sub_2026,
    orientation='h',
    title='Africa sub-regions: mean overall score 2026',
    labels={'value': 'Mean overall score (0-100)', 'subregion': 'Sub-region'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Sub-region means, 2026:
subregion
Northern Africa    23.17
Middle Africa      23.56
Eastern Africa     35.78
Western Africa     48.38
Southern Africa    62.60


In [10]:
s2013 = africa_total[africa_total['Year'] == 2013].groupby('subregion')['Score'].mean()
s2026 = africa_total[africa_total['Year'] == 2026].groupby('subregion')['Score'].mean()
sub_change = (s2026 - s2013).round(2).sort_values()
print('Change 2013 to 2026:')
print(sub_change.to_string())

fig = px.bar(
    sub_change,
    orientation='h',
    title='Africa sub-regions: change in mean overall score, 2013 to 2026',
    labels={'value': 'Change in score', 'subregion': 'Sub-region'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Change 2013 to 2026:
subregion
Northern Africa   -14.83
Middle Africa      -7.67
Eastern Africa     -6.17
Western Africa     -4.31
Southern Africa    -2.20


### Interpretation

The 2026 African ranking: **Southern 62.6, Western 48.4, Eastern 35.8, Middle 23.6, Northern 23.2.** Northern Africa's collapse (−14.8) is by far the continent's biggest — driven by its low scorers (Libya, Sudan) that notebook 04 flagged among the world's deepest decliners. Eastern Africa's decline (−6.2) sits mid-pack, but its level is now third of five.

## 8. Region × category heatmap (2026)

**Question:** do regions differ in *which* rights are strong or weak?

**Method:** mean score per region as % of scale for the seven categories plus PR, CL and TOTAL in 2026 (the same normalized view as notebook 05, section 10.2).

In [11]:
cats = ['FH_FIW_A', 'FH_FIW_B', 'FH_FIW_C', 'FH_FIW_D', 'FH_FIW_E', 'FH_FIW_F', 'FH_FIW_G', 'FH_FIW_PR', 'FH_FIW_CL', 'FH_FIW_TOTAL']
cat_labels = {'FH_FIW_A': 'A Electoral', 'FH_FIW_B': 'B Pluralism', 'FH_FIW_C': 'C Gov. function', 'FH_FIW_D': 'D Expression', 'FH_FIW_E': 'E Association', 'FH_FIW_F': 'F Rule of law', 'FH_FIW_G': 'G Personal autonomy', 'FH_FIW_PR': 'PR total', 'FH_FIW_CL': 'CL total', 'FH_FIW_TOTAL': 'Overall'}
sm_map = {'0_TO_4': 4, '0_TO_12': 12, '0_TO_16': 16, '0_TO_40': 40, '0_TO_60': 60, '0_TO_100': 100}

region_cat = long[long['INDICATOR'].isin(cats) & (long['Year'] == 2026)].copy()
region_cat['Score'] = pd.to_numeric(region_cat['Score'], errors='coerce')
region_cat['pct'] = region_cat['Score'] / region_cat['UNIT_MEASURE'].map(sm_map) * 100
region_cat['region'] = region_cat['Economy'].map(region_lookup)

pivot = region_cat.pivot_table(index='region', columns='INDICATOR', values='pct', aggfunc='mean')
pivot = pivot[list(cat_labels.keys())].rename(columns=cat_labels).round(1)
print(pivot.to_string())

fig = create_heatmap(
    pivot,
    title='Region x category: mean score as % of scale, 2026',
    colorbar_label='% of scale',
)
fig.show()

INDICATOR  A Electoral  B Pluralism  C Gov. function  D Expression  E Association  F Rule of law  G Personal autonomy  PR total  CL total  Overall
region                                                                                                                                            
Africa            35.8         37.6             30.6          50.2           43.7           31.9                 40.0      34.4      41.3     38.6
Americas          76.7         75.2             61.0          81.4           74.8           56.6                 69.6      71.4      70.3     70.7
Asia              37.2         37.0             33.5          40.4           36.9           31.5                 44.6      35.2      38.5     37.1
Europe            85.2         85.2             73.9          83.2           86.6           74.7                 83.2      81.8      81.6     81.7
Oceania           86.3         89.7             71.4          93.3           90.5           79.0                 78.1 

### Interpretation

The heatmap shows the two-tier world in component detail: **Europe and Oceania fill their cells at 70–90%**, the **Americas at 55–85%**, while **Asia and Africa sit at 30–50%**. The weakest category in every region is **rule of law (F)** — lowest in Africa (31.9%) and Asia (31.5%), highest in Oceania (79.0%). The strongest category is **expression (D)** in most regions — except Asia, where personal autonomy (G, 44.6%) leads. Regional differences are differences of degree, not of kind: the same categories are strong and weak everywhere, but at very different levels.

## Summary and next question

### What we learned

- **Two-tier world in 2026**: Oceania (84.2), Europe (81.7) and the Americas (70.7) versus Asia (37.1) and Africa (38.6) — a ~35-point gap.
- **One exception to the global decline**: Oceania rose (+2.6) while Africa (−6.5), the Americas (−4.8), Asia (−4.4) and Europe (−3.1) all fell.
- **Inside Africa**: Southern Africa leads (62.6) and declined least (−2.2); **Northern Africa collapsed (−14.8 to 23.2)**; Eastern Africa — home of the EAC — fell to 35.8.
- **Components**: the same categories lead everywhere (rule of law weakest, expression strongest in most regions), but at very different levels — Asia and Africa sit 30–50% of scale versus 70–90% for Europe and Oceania.
- All classifications are the documented UN M49 scheme applied to the dataset's own labels; nothing is assumed.

### Next question

*What do the 40 indicators say individually?* — notebook 07, indicator analysis, examines the strongest and weakest dimensions, the 1–7 ratings and the categorical status across economies and time.